In [ ]:
import os
import sys
import math
import random
import time
import scipy
import json
from pathlib import Path
from typing import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import transformers
import xgboost as xgb

import torch
import tensorflow as tf

SEED = 1432

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

DEVICE = "mps"

tf.config.set_visible_devices([], "CPU")

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"

for d in [DATA_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

np.set_printoptions(suppress=True, precision=4)

In [2]:
import kagglehub
import os
import cv2
from utils import sketch

path = kagglehub.dataset_download("wenewone/cub2002011") + "/CUB_200_2011"


Resuming download from 738197504 bytes (866096906 bytes left)...
Resuming download to /Users/jacobpercy/.cache/kagglehub/datasets/wenewone/cub2002011/7.archive (738197504/1604294410) bytes left.


100%|██████████| 1.49G/1.49G [00:25<00:00, 33.8MB/s]

Extracting files...


In [3]:
destination_imgs = "/Users/jacobpercy/Desktop/birds2/images"
destination_sketches = "/Users/jacobpercy/Desktop/birds2/sketches"

os.makedirs(destination_imgs, exist_ok=True)
os.makedirs(destination_sketches, exist_ok=True)

for folder in os.listdir(os.path.join(path, "images")):
    folder_path = os.path.join(path, "images", folder)
    
    for file in os.listdir(folder_path):
        src_path = os.path.join(folder_path, file)
        dst_img = os.path.join(destination_imgs, file)
        dst_sket = os.path.join(destination_sketches, file)

        if os.path.isfile(src_path):
            img = cv2.imread(src_path)
            
            if img is not None:
                sketched = sketch(img)
                cv2.imwrite(dst_sket, sketched)
                
                #move
                os.rename(src_path, dst_img)
            else:
                print(f"Failed to read: {src_path}")

In [4]:
import os
import random
import shutil

destination = "/Users/jacobpercy/Desktop/birds2"
images_source = os.path.join(destination, "images")
sketches_source = os.path.join(destination, "sketches")

train_dir = os.path.join(destination, "train")
test_dir = os.path.join(destination, "test")

for sub in ["images", "sketches"]:
    os.makedirs(os.path.join(train_dir, sub), exist_ok=True)
    os.makedirs(os.path.join(test_dir, sub), exist_ok=True)

paired_files = []
for filename in os.listdir(images_source):
    image_path = os.path.join(images_source, filename)
    sketch_path = os.path.join(sketches_source, filename)
    if os.path.isfile(image_path) and os.path.isfile(sketch_path):
        paired_files.append(filename)

random.shuffle(paired_files)
split_index = int(len(paired_files) * 0.9)
train_files = paired_files[:split_index]
test_files = paired_files[split_index:]

for filename in train_files:
    shutil.copy(os.path.join(images_source, filename),
                os.path.join(train_dir, "images", filename))
    shutil.copy(os.path.join(sketches_source, filename),
                os.path.join(train_dir, "sketches", filename))

for filename in test_files:
    shutil.copy(os.path.join(images_source, filename),
                os.path.join(test_dir, "images", filename))
    shutil.copy(os.path.join(sketches_source, filename),
                os.path.join(test_dir, "sketches", filename))

print("done")


done
